In [ ]:
import numpy as np
import scanpy as sc

from collections import Counter
from sklearn.neighbors import NearestNeighbors

In [ ]:
adata = sc.read('./Perturb-FISH-rawadata/Measured_annotated.h5ad')

adata

# Build niche and microenvironment keys

In [ ]:
def build_niche_and_microenv_keys(
    adata,
    K=10,
    perturb_key="perturb",
    celltype_key="cell_type",
    spatial_key="spatial",
):
    """
    Add spatial niche and microenvironment annotations to adata.obs.

    Added columns
    -------------
    perturb_celltype_key
        Central perturbation and cell type:
        perturb|cell_type

    microenv_key
        Composition of perturbation-cell-type labels among the K
        nearest spatial neighbors:
        label1:count1;label2:count2;...

    celltype_microenv_key
        Central cell type combined with the microenvironment.

    niche_key
        Central perturbation, central cell type, and microenvironment.
    """

    if spatial_key not in adata.obsm:
        raise KeyError(f"`{spatial_key}` not found in adata.obsm.")

    coords = np.asarray(adata.obsm[spatial_key])

    if coords.shape[0] != adata.n_obs:
        raise ValueError(
            "Number of spatial coordinates does not match adata.n_obs."
        )

    perturb = adata.obs[perturb_key].astype(str).to_numpy()
    celltype = adata.obs[celltype_key].astype(str).to_numpy()

    perturb_celltype = np.array(
        [f"{p}|{ct}" for p, ct in zip(perturb, celltype)],
        dtype=object,
    )

    adata.obs["perturb_celltype_key"] = perturb_celltype

    nn = NearestNeighbors(
        n_neighbors=K + 1,
        metric="euclidean",
    )
    nn.fit(coords)

    neighbor_indices = nn.kneighbors(
        coords,
        return_distance=False,
    )[:, 1:]

    microenv_keys = []

    for idx in neighbor_indices:
        counts = Counter(perturb_celltype[idx])

        composition = ";".join(
            f"{label}:{counts[label]}"
            for label in sorted(counts)
        )

        microenv_keys.append(composition)

    microenv_keys = np.asarray(
        microenv_keys,
        dtype=object,
    )

    adata.obs["microenv_key"] = microenv_keys

    adata.obs["celltype_microenv_key"] = [
        f"{ct}|{env}"
        for ct, env in zip(
            celltype,
            microenv_keys,
        )
    ]

    adata.obs["niche_key"] = [
        f"{p}|{ct}|{env}"
        for p, ct, env in zip(
            perturb,
            celltype,
            microenv_keys,
        )
    ]

    return adata

In [ ]:
adata = build_niche_and_microenv_keys(
    adata,
    K=6,
    perturb_key="perturb",
    celltype_key="cell_type",
    spatial_key="spatial",
)

In [ ]:
adata.obs[
    [
        "perturb",
        "cell_type",
        "perturb_celltype_key",
        "microenv_key",
        "celltype_microenv_key",
        "niche_key",
    ]
].head()

,perturb,cell_type,perturb_celltype_key,microenv_key,celltype_microenv_key,niche_key
Cell_ID,,,,,,
Cell_15,None,T_cell,None|T_cell,None|T_cell:6,T_cell|None|T_cell:6,None|T_cell|None|T_cell:6
Cell_21,None,T_cell,None|T_cell,MAP2K4|Tumor:1;None|T_cell:3;None|Tumor:2,T_cell|MAP2K4|Tumor:1;None|T_cell:3;None|Tumor:2,None|T_cell|MAP2K4|Tumor:1;None|T_cell:3;None|...
Cell_22,None,T_cell,None|T_cell,MAP2K4|Tumor:1;None|T_cell:2;None|Tumor:3,T_cell|MAP2K4|Tumor:1;None|T_cell:2;None|Tumor:3,None|T_cell|MAP2K4|Tumor:1;None|T_cell:2;None|...
Cell_25,None,Tumor,None|Tumor,None|Tumor:6,Tumor|None|Tumor:6,None|Tumor|None|Tumor:6
Cell_29,None,T_cell,None|T_cell,None|T_cell:1;None|Tumor:5,T_cell|None|T_cell:1;None|Tumor:5,None|T_cell|None|T_cell:1;None|Tumor:5


# OOD split

In [19]:
def add_train_ood_split_with_niche_masking(
    adata,
    ood_perturbs,
    perturb_key="perturb",
    spatial_key="spatial",
    K=6,
    split_key="split",
):
    """
    Define perturbation OOD cells and mask their spatial neighbors.

    OOD perturbation cells are labeled as "ood".
    Train cells among the K nearest neighbors of any OOD cell
    are labeled as "ood_neighbor".
    """

    if spatial_key not in adata.obsm:
        raise KeyError(f"`{spatial_key}` not found in adata.obsm.")

    coords = np.asarray(adata.obsm[spatial_key])

    if coords.shape[0] != adata.n_obs:
        raise ValueError(
            "Number of spatial coordinates does not match adata.n_obs."
        )

    perturb = adata.obs[perturb_key].astype(str).to_numpy()
    ood_perturbs = set(map(str, ood_perturbs))

    ood_mask = np.isin(
        perturb,
        list(ood_perturbs),
    )

    split = np.full(
        adata.n_obs,
        "train",
        dtype=object,
    )
    split[ood_mask] = "ood"

    nn = NearestNeighbors(
        n_neighbors=K + 1,
        metric="euclidean",
    )
    nn.fit(coords)

    neighbor_indices = nn.kneighbors(
        coords,
        return_distance=False,
    )[:, 1:]

    # Same logic as the original nested loop:
    # collect neighbors of OOD cells and mask only current train cells.
    ood_neighbor_idx = np.unique(
        neighbor_indices[ood_mask].ravel()
    )

    train_neighbor_idx = ood_neighbor_idx[
        split[ood_neighbor_idx] == "train"
    ]

    split[train_neighbor_idx] = "ood_neighbor"

    adata.obs[split_key] = split
    adata.obs[split_key] = (
        adata.obs[split_key]
        .replace("ood_neighbor", "ood")
    )
    return adata

In [20]:
def add_validation_split(
    adata,
    split_key,
    val_fraction=0.2,
    seed=42,
):
    """
    Randomly assign a fraction of training cells to validation.
    """

    train_idx = np.flatnonzero(
        adata.obs[split_key].to_numpy() == "train"
    )

    n_val = int(
        val_fraction * len(train_idx)
    )

    np.random.seed(seed)

    val_idx = np.random.choice(
        train_idx,
        size=n_val,
        replace=False,
    )

    col_idx = adata.obs.columns.get_loc(split_key)

    adata.obs.iloc[
        val_idx,
        col_idx,
    ] = "val"

    return adata

In [24]:
ood_perturbs = ["MAP3K7",'JUN']

In [25]:
adata = add_train_ood_split_with_niche_masking(
    adata,
    ood_perturbs=ood_perturbs,
    perturb_key="perturb",
    spatial_key="spatial",
    K=6,
    split_key="split",
)

In [26]:
adata.obs["split"].value_counts()

split
train    120251
ood         343
Name: count, dtype: int64

In [28]:
adata = add_validation_split(
    adata,
    split_key="split",
    val_fraction=0.2,
    seed=42,
)

In [29]:
adata.obs["split"].value_counts()

split
train    96201
val      24050
ood        343
Name: count, dtype: int64

In [30]:
adata.write_h5ad('./Perturb-FISH-rawadata/Measured_annotated_splited.h5ad')